# 01. Geneformer — rank value encoding을 직접 구현하고 임베딩 뽑기

**등급**: 🟢 Mac 로컬 (MPS/CPU) · **예상 시간**: Mac MPS에서 약 20분, Colab T4에서 약 17분 (V2-104M 임베딩이 대부분) · **다운로드**: PBMC H5AD 약 217 MB, Geneformer V1-10M 약 41 MB + V2-104M 약 418 MB, 유전자 사전 pkl 합 약 8 MB

**질문**: Geneformer는 세포를 "발현량 벡터"가 아니라 "유전자 순위 목록"으로 읽는다. 그 입력을 우리가 직접 만들 수 있는가? 그 토큰 열에는 무엇이 남고 무엇이 사라지는가? 사전학습 가중치에서 뽑은 세포 임베딩은 어떻게 얻는가?

**이 노트북이 하는 일**
1. 데이터(Wilk et al. 2020 PBMC, 13명)를 점검하고 **기증자당 최대 800개**로 균형 하위표본(≈10k 세포)을 만든다.
2. 유전자 사전 `.pkl`을 **허용 목록 Unpickler**로만 읽는다 (pickle은 로드 시 코드를 실행할 수 있다).
3. 공식 `TranscriptomeTokenizer`의 로직(중앙값 정규화 → 순위 → 절단)을 **직접 구현**하고 성질을 검사한다.
4. V1-10M / V2-104M / (V1 무작위 초기화 대조군)에서 세포 임베딩을 추출해 저장하며, 노트북 02·03에서 쓴다.
5. 마스크 유전자 예측으로 "사전학습이 실제로 무언가 배웠는가"를 간단히 점검한다.

> 공식 `geneformer` 패키지는 설치하지 않았다. 여기 구현은 공식 소스(`tokenizer.py`, `emb_extractor.py`)를 읽고 다시 쓴 것이며, **공식 패키지 출력과 토큰 단위로 대조하지는 못했다** (패키지 미설치). 동점 처리는 재현성을 위해 안정 정렬을 썼다 (공식은 `np.argsort` 기본값).

In [1]:
import hashlib
import io
import os
import pickle
import pickletools
import time
import urllib.request
import warnings
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")


def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "SETUP.md").exists():
            return q
    return None


ROOT = find_root()
CACHE = (ROOT / ".cache") if ROOT is not None else Path("cache")
os.environ.setdefault("HF_HOME", str(CACHE / "hf"))
SC = CACHE / "singlecell"
SC.mkdir(parents=True, exist_ok=True)

import torch
from huggingface_hub import hf_hub_download
from transformers import BertConfig, BertForMaskedLM, BertModel
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
import importlib.metadata as md

print("device:", DEVICE, "| cache:", CACHE)
print({k: md.version(k) for k in ["numpy", "scipy", "pandas", "anndata", "torch", "transformers", "huggingface_hub"]})

W1110 11:57:59.896000 26630 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: mps | cache: <repo>/.cache
{'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.1', 'anndata': '0.13.4', 'torch': '2.15.0', 'transformers': '5.22.1', 'huggingface_hub': '2.4.0'}


## 1. 데이터 내려받기와 무결성 확인
CELLxGENE Discover의 데이터셋 버전 URL(버전 ID가 URL에 들어 있어 내용이 바뀌지 않는다). 이미 받아 두었으면 건너뛴다.
- Wilk et al., *Nat Med* 2020, "A single-cell atlas of the peripheral immune response in patients with severe COVID-19" (DOI 10.1038/s41591-020-0944-y), Seq-Well, PBMC.
- 라이선스는 CELLxGENE 데이터 정책(컬렉션 페이지에서 확인)을 따른다. 이 환경에서 컬렉션 API로는 라이선스 필드를 확인하지 못했으므로 **재배포하지 않고** 로컬 캐시로만 쓴다.

In [2]:
H5AD_URL = "https://datasets.cellxgene.cziscience.com/8d7fad46-2f37-4aa0-ac9d-babbc36efb6f.h5ad"
H5AD_SHA256 = "50fa832f6ce824a18f1a5d54d2481ada5bee24036630a56bac7f7e98123a7c16"
H5AD = SC / "wilk2020_pbmc.h5ad"


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()


if not H5AD.exists():
    print("downloading ~217 MB ...")
    urllib.request.urlretrieve(H5AD_URL, H5AD.with_suffix(".part"))
    H5AD.with_suffix(".part").rename(H5AD)
assert sha256(H5AD) == H5AD_SHA256, "내려받은 H5AD의 sha256이 기록된 값과 다르다 — 데이터가 바뀌었거나 손상되었다"
print("sha256 일치 | %.0f MB" % (H5AD.stat().st_size / 1e6))

adata = ad.read_h5ad(H5AD)          # HDF5 읽기 (pickle 아님)
print(adata)

sha256 일치 | 217 MB


AnnData object with n_obs × n_vars = 44721 × 24505
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'percent.rps', 'percent.rpl', 'percent.rrna', 'nCount_SCT', 'nFeature_SCT', 'seurat_clusters', 'singler', 'cell.type.fine', 'cell.type.coarse', 'IFN1', 'HLA1', 'donor_id', 'DPS', 'DTF', 'Admission', 'Ventilated', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_pca', 'X_umap'
    layers: None (

### 1.1 데이터 점검 — 무엇을 믿고 무엇을 의심할지
- `raw.X`가 정수 카운트인가, 유전자 ID가 Ensembl인가 (Geneformer는 **Ensembl ID**로 토큰을 찾는다).
- 기증자와 질병 상태가 얼마나 겹치는가 (**배치 = 기증자**로 쓸 때의 교란).
- 제공된 `nCount_RNA`가 행렬의 합과 일치하는가 (Geneformer의 `n_counts` 정규화에 쓰인다).

In [3]:
X_raw = adata.raw.X.tocsr()
ids = adata.raw.var_names
sample = X_raw.data[:200000]
print("raw.X 정수형:", bool(np.allclose(sample, np.round(sample))), "| 최소 비영 값:", X_raw.data.min(), "| 최대:", X_raw.data.max())
print("유전자 ID 예:", list(ids[:3]), "| 중복:", int(ids.duplicated().sum()), "| 버전 접미사('.'):", int(ids.str.contains(r'\.').sum()))

tot = np.asarray(X_raw.sum(1)).ravel()
ngen = np.diff(X_raw.indptr)
print(f"세포당 총 카운트 중앙값 {np.median(tot):.0f} [5%,95%: {np.percentile(tot, 5):.0f}, {np.percentile(tot, 95):.0f}] | 검출 유전자 중앙값 {np.median(ngen):.0f}")
r = tot / adata.obs["nCount_RNA"].values
print(f"raw.X 합 / 제공 nCount_RNA: 중앙값 {np.median(r):.2f}, 5~95% [{np.percentile(r, 5):.2f}, {np.percentile(r, 95):.2f}], 상관계수 {np.corrcoef(tot, adata.obs['nCount_RNA'])[0, 1]:.2f}")
print("→ 제공된 nCount_RNA는 이 행렬의 합과 맞지 않는다(다른 유전자 집합/처리에서 계산된 값으로 보임, 미확인). 아래에서는 raw.X의 합을 n_counts로 쓴다.\n")

dd = adata.obs.groupby("donor_id", observed=True).agg(disease=("disease", "first"), n=("disease", "size"))
print(dd.T.to_string())
print("\n기증자별 질병 종류 수의 최댓값:", int(adata.obs.groupby("donor_id", observed=True)["disease"].nunique().max()), "→ 질병 상태는 기증자와 완전히 겹친다(교란).")

raw.X 정수형: True | 최소 비영 값: 1.0 | 최대: 1718.0
유전자 ID 예: ['ENSG00000235399', 'ENSG00000151338', 'ENSG00000182173'] | 중복: 0 | 버전 접미사('.'): 0
세포당 총 카운트 중앙값 1508 [5%,95%: 1003, 2222] | 검출 유전자 중앙값 919
raw.X 합 / 제공 nCount_RNA: 중앙값 0.79, 5~95% [0.24, 1.10], 상관계수 0.53
→ 제공된 nCount_RNA는 이 행렬의 합과 맞지 않는다(다른 유전자 집합/처리에서 계산된 값으로 보임, 미확인). 아래에서는 raw.X의 합을 n_counts로 쓴다.

donor_id        C1        C2        C3        C4        C5        C6        C7      H1      H2      H3      H4      H5      H6
disease   COVID-19  COVID-19  COVID-19  COVID-19  COVID-19  COVID-19  COVID-19  normal  normal  normal  normal  normal  normal
n             8362      1735      6545      3688      2507      1893      3364    2100    1941    2306    5136    3146    1998

기증자별 질병 종류 수의 최댓값: 1 → 질병 상태는 기증자와 완전히 겹친다(교란).


### 1.2 세포 유형 라벨과 균형 하위표본
CELLxGENE의 `cell_type`(온톨로지)은 입도가 섞여 있다(예: `monocyte` 대 `CD14-positive monocyte`). 원 논문의 거친 분류 `cell.type.coarse`(13종)를 **정답 라벨**로 쓴다. 이 라벨은 논문 저자가 클러스터링 후 붙인 것이므로 사람이 만든 기준일 뿐 절대 정답은 아니다.

기증자당 최대 800개를 무작위(seed 0)로 뽑아 C1(8,362개) 같은 큰 기증자가 지배하지 않게 한다. 작은 세포 유형(pDC 등)은 개수가 적어 지표의 불확실성이 크다.

In [4]:
CAP = 800


def balanced_subsample(obs, group="donor_id", cap=CAP, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = []
    for _, g in obs.groupby(group, observed=True).indices.items():
        idx += list(rng.choice(g, size=min(cap, len(g)), replace=False))
    return np.sort(np.array(idx))


sel = balanced_subsample(adata.obs)
obs = adata.obs.iloc[sel][["donor_id", "disease", "cell.type.coarse", "cell_type"]].copy()
obs.columns = ["donor", "disease", "label", "cell_type_cxg"]
for c in obs.columns:
    obs[c] = obs[c].astype(str)
counts = X_raw[sel].tocsr()
n_counts = np.asarray(counts.sum(1)).ravel()
obs["n_counts"] = n_counts
print("하위표본:", counts.shape, "| 기증자 수", obs.donor.nunique())
print(pd.crosstab(obs.label, obs.disease).assign(합=lambda d: d.sum(1)).sort_values("합", ascending=False).to_string())

하위표본: (10400, 24505) | 기증자 수 13
disease        COVID-19  normal     합
label                                
CD14 Monocyte      1460     692  2152
CD4 T              1129     857  1986
NK                  485    1058  1543
CD8 T               582     918  1500
B                   692     598  1290
RBC                 481      82   563
PB                  393      21   414
CD16 Monocyte       107     259   366
Granulocyte         120      54   174
gd T                  3     136   139
DC                   51      57   108
Platelet             79      23   102
pDC                  18      45    63


## 2. 유전자 사전 `.pkl` — 안전하게 읽기
Geneformer 저장소는 토큰 사전, 유전자별 코퍼스 중앙값, Ensembl 매핑을 **pickle**로 배포한다. pickle은 불러오는 순간 임의 코드를 실행시킬 수 있다. 신뢰할 수 있는 저장소라도 (1) 어떤 전역 객체를 끌어오는지 **불러오기 전에** 보고, (2) 필요한 클래스만 허용하는 `Unpickler`로 읽는다.

아래 `opcode` 검사는 사람이 읽기 위한 점검용이다(STACK_GLOBAL 직전 문자열 두 개를 보는 근사라 모든 경우를 잡는다고 보장하지 않는다). **실제 방어는 `find_class` 허용 목록**이다.

In [5]:
REV = "1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5"      # 이 노트북에서 쓴 저장소 커밋 (재현용 고정)
REPO = "ctheodoris/Geneformer"
FILES = {
    "v1": dict(model="Geneformer-V1-10M", tok="geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl",
               med="geneformer/gene_dictionaries_30m/gene_median_dictionary_gc30M.pkl",
               map="geneformer/gene_dictionaries_30m/ensembl_mapping_dict_gc30M.pkl", max_len=2048, special=False),
    "v2": dict(model="Geneformer-V2-104M", tok="geneformer/token_dictionary_gc104M.pkl",
               med="geneformer/gene_median_dictionary_gc104M.pkl",
               map="geneformer/ensembl_mapping_dict_gc104M.pkl", max_len=4096, special=True),
}


def fetch(rel):
    return hf_hub_download(REPO, rel, revision=REV)


def audit_pickle_globals(path):
    # 근사 점검: STACK_GLOBAL 직전 문자열 두 개를 보므로 MEMOIZE/BINGET으로 참조된 문자열, INST/OBJ/NEWOBJ_EX 등은 추적하지 못한다. 따라서 보조 출력일 뿐 방어 수단이 아니다
    data = Path(path).read_bytes()
    seen, strs = set(), []
    for op, arg, _ in pickletools.genops(data):
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE"):
            strs.append(arg)
        elif op.name == "GLOBAL":
            seen.add(arg)
        elif op.name == "STACK_GLOBAL":
            seen.add(" ".join(strs[-2:]))
    return sorted(seen)


class SafeUnpickler(pickle.Unpickler):
    ALLOWED = {("numpy", "dtype"), ("numpy.core.multiarray", "scalar"), ("numpy._core.multiarray", "scalar")}

    def find_class(self, module, name):
        if (module, name) in self.ALLOWED:
            return super().find_class(module, name)
        raise pickle.UnpicklingError(f"blocked global: {module}.{name}")


def load_pkl(path):
    with open(path, "rb") as f:
        return SafeUnpickler(f).load()


for v, cfg in FILES.items():
    for kind in ["tok", "med", "map"]:
        p = fetch(cfg[kind])
        print(f"{v} {kind:3s} {Path(cfg[kind]).name:42s} 참조하는 전역 객체: {audit_pickle_globals(p) or '없음(내장 자료형뿐)'}")

# 허용 목록이 실제로 막는지: 무해한 print를 실행하려는 가짜 pickle
class Evil:
    def __reduce__(self):
        return (print, ("!!! 코드가 실행되었다 !!!",))


evil = pickle.dumps(Evil())
try:
    SafeUnpickler(io.BytesIO(evil)).load()
except pickle.UnpicklingError as e:
    print("\n악성 모의 pickle 차단됨 →", e)

v1 tok token_dictionary_gc30M.pkl                 참조하는 전역 객체: ['numpy dtype', 'numpy.core.multiarray scalar']


v1 med gene_median_dictionary_gc30M.pkl           참조하는 전역 객체: ['numpy dtype', 'numpy.core.multiarray scalar']


v1 map ensembl_mapping_dict_gc30M.pkl             참조하는 전역 객체: 없음(내장 자료형뿐)
v2 tok token_dictionary_gc104M.pkl                참조하는 전역 객체: 없음(내장 자료형뿐)


v2 med gene_median_dictionary_gc104M.pkl          참조하는 전역 객체: ['numpy dtype', 'numpy.core.multiarray scalar']


v2 map ensembl_mapping_dict_gc104M.pkl            참조하는 전역 객체: 없음(내장 자료형뿐)

악성 모의 pickle 차단됨 → blocked global: builtins.print


## 3. Rank value encoding 직접 구현
공식 `TranscriptomeTokenizer.tokenize_anndata`가 하는 일을 순서대로 쓰면 다음과 같다.
1. 유전자 ID를 매핑 사전으로 **공식 Ensembl ID에 합친다** (같은 ID로 모이는 열은 카운트 합산).
2. 어휘에 있는 유전자(V1은 단백질 코딩+miRNA 등 25,424개, V2는 단백질 코딩 중심 20,271개)만 남긴다.
3. 각 세포를 `count / n_counts × 10,000`으로 **깊이 정규화**한다.
4. 유전자별 **코퍼스 중앙값**(사전학습 코퍼스에서 비영 값의 중앙값)으로 나눈다. 그러면 어디서나 높게 발현되는 하우스키핑 유전자는 낮아지고, 드물게 켜지는 조절 유전자는 올라간다.
5. 0이 아닌 유전자만 값 내림차순으로 정렬하여 **토큰 ID 열**로 만든다.
6. 입력 길이에 맞춰 앞에서부터 자른다. V2는 앞뒤에 `<cls>`, `<eos>`를 붙인다 (V1은 특수 토큰 없음).

In [6]:
class RankTokenizer:
    def __init__(self, version):
        cfg = FILES[version]
        self.cfg = cfg
        self.tok = load_pkl(fetch(cfg["tok"]))
        self.med = load_pkl(fetch(cfg["med"]))
        mp = load_pkl(fetch(cfg["map"]))
        self.map = {k: v for k, v in mp.items() if v in self.tok}      # 공식 구현과 같이 어휘에 있는 값만 유지
        self.genes = [g for g in self.tok if g.startswith("ENS")]
        self.gidx = {g: i for i, g in enumerate(self.genes)}
        self.token_ids = np.array([self.tok[g] for g in self.genes])
        self.median = np.array([self.med[g] for g in self.genes])
        self.max_len, self.special = cfg["max_len"], cfg["special"]

    def collapse(self, counts, var_ids):
        mapped = pd.Series(np.asarray(var_ids)).str.upper().map(self.map)
        keep = np.where(mapped.notna().values)[0]
        cols = [self.gidx[mapped.iloc[i]] for i in keep]
        G = sp.csr_matrix((np.ones(len(keep)), (np.arange(len(keep)), cols)), shape=(len(keep), len(self.genes)))
        X = (counts[:, keep] @ G).tocsr()
        X.eliminate_zeros()                                   # 명시적 0이 토큰이 되지 않도록
        return X, len(keep)

    def normalize_collapsed(self, Xc, n_counts, target_sum=10_000, use_median=True):
        Xn = sp.diags(target_sum / n_counts) @ Xc
        if use_median:
            Xn = Xn @ sp.diags(1.0 / self.median)
        return Xn.tocsr()

    def normalized(self, counts, var_ids, n_counts, **kw):
        Xc, n_keep = self.collapse(counts, var_ids)
        return self.normalize_collapsed(Xc, n_counts, **kw), n_keep

    def to_tokens(self, Xn, Xc=None):
        """Xc(합쳐진 원시 카운트)를 주면 토큰과 같은 순서의 카운트 배열도 돌려준다 (공식 keep_counts에 해당)."""
        if Xc is not None:
            assert np.array_equal(Xn.indptr, Xc.indptr) and np.array_equal(Xn.indices, Xc.indices)
        L = self.max_len - (2 if self.special else 0)
        out, cnt = [], []
        for i in range(Xn.shape[0]):
            s, e = Xn.indptr[i], Xn.indptr[i + 1]
            v, ix = Xn.data[s:e], Xn.indices[s:e]
            order = np.argsort(-v, kind="stable")
            t = self.token_ids[ix[order]][:L]
            if self.special:
                t = np.concatenate([[self.tok["<cls>"]], t, [self.tok["<eos>"]]])
            out.append(t.astype(np.int64))
            if Xc is not None:
                c = Xc.data[s:e][order][:L]
                cnt.append((np.concatenate([[0], c, [0]]) if self.special else c).astype(np.float32))
        return (out, cnt) if Xc is not None else out

    def __call__(self, counts, var_ids, n_counts, **kw):
        Xn, n_keep = self.normalized(counts, var_ids, n_counts, **kw)
        return self.to_tokens(Xn), n_keep


var_ids = adata.raw.var_names.values
tk = {v: RankTokenizer(v) for v in FILES}
tokens, tok_stats = {}, {}
for v, t in tk.items():
    t0 = time.time()
    tokens[v], n_keep = t(counts, var_ids, n_counts)
    L = np.array([len(x) for x in tokens[v]])
    assert L.min() > (2 if t.special else 0), "어휘 유전자가 하나도 없는 세포가 있다"
    tok_stats[v] = dict(어휘_유전자=len(t.genes), 데이터에서_어휘와_연결된_열=n_keep, 길이_평균=L.mean(), 길이_중앙값=np.median(L), 길이_최대=L.max(),
                        절단된_세포_비율=(L >= t.max_len).mean(), 초=time.time() - t0)
print(pd.DataFrame(tok_stats).round(3).to_string())

cov = {}
tot_all = np.asarray(X_raw.sum(0)).ravel()
for v, t in tk.items():
    m = pd.Series(np.asarray(ids)).str.upper().map(t.map).notna().values
    cov[v] = dict(어휘와_연결된_유전자_비율=m.mean(), 연결된_유전자가_가진_카운트_비율=tot_all[m].sum() / tot_all.sum())
print("\n", pd.DataFrame(cov).round(3).to_string())

                        v1         v2
어휘_유전자           25424.000  20271.000
데이터에서_어휘와_연결된_열  17393.000  17061.000
길이_평균              911.068    912.284
길이_중앙값             875.000    876.000
길이_최대             1873.000   1873.000
절단된_세포_비율            0.000      0.000
초                    0.821      0.805

                        v1     v2
어휘와_연결된_유전자_비율      0.710  0.696
연결된_유전자가_가진_카운트_비율  0.896  0.896


### 3.1 토큰화 성질 검사 (단위 점검)
구현이 의도대로인지 확인하고, 이 표현이 **무엇을 버리는지**를 수치로 본다.
1. 열 순서를 섞어도 같은 토큰열인가 (유전자 순서 불변).
2. 세포의 카운트를 2배 해도 같은가 (깊이 정규화로 **스케일 불변**).
3. 값의 크기는 버려지고 **순서만** 남는다. 관측된 카운트 분포에서 얼마나 많은 유전자가 카운트 1인가? Seq-Well처럼 깊이가 낮은 데이터에서는 카운트 1 유전자들의 상대 순서가 **세포와 무관한 중앙값 순서**로 정해진다.

In [7]:
t1 = tk["v1"]
sub5 = counts[:5]
base = t1(sub5, var_ids, n_counts[:5])[0]
perm = np.random.default_rng(1).permutation(len(var_ids))
shuf = t1(sub5[:, perm], var_ids[perm], n_counts[:5])[0]
dbl = t1(sub5 * 2, var_ids, n_counts[:5] * 2)[0]
print("열 순서를 섞어도 동일:", all(np.array_equal(a, b) for a, b in zip(base, shuf)))
print("카운트 2배(n_counts도 2배)해도 동일:", all(np.array_equal(a, b) for a, b in zip(base, dbl)))
dbl_only = t1(sub5 * 2, var_ids, n_counts[:5])[0]            # n_counts를 갱신하지 않으면 절대 규모가 바뀌지만 순위는 같다
print("카운트만 2배(n_counts 고정)해도 순위 동일:", all(np.array_equal(a, b) for a, b in zip(base, dbl_only)))

# 카운트 분포: 검출 유전자 중 카운트 1의 비율
Xc, _ = t1.collapse(counts, var_ids)
nz = Xc.data
print(f"\n하위표본의 비영 항목 중 카운트=1 비율: {np.mean(nz == 1):.2f}, 카운트=2: {np.mean(nz == 2):.2f}, 카운트≥3: {np.mean(nz >= 3):.2f}")

# 카운트=1 유전자끼리의 순서가 세포와 무관한가: 세포마다 토큰열을 만든 뒤, 카운트=1인 유전자만 뽑아 중앙값이 오름차순인지 확인
frac_sorted, frac_ones_in_cell = [], []
for i in range(300):
    s_, e_ = Xc.indptr[i], Xc.indptr[i + 1]
    ix, v = Xc.indices[s_:e_], Xc.data[s_:e_]
    order = np.argsort(-(v / t1.median[ix]), kind="stable")
    ones_ranked = ix[order][v[order] == 1]
    frac_sorted.append(bool(np.all(np.diff(t1.median[ones_ranked]) >= 0)))
    frac_ones_in_cell.append(len(ones_ranked) / len(ix))
print("세포 안에서 카운트=1 유전자의 비율 (중앙값):", round(float(np.median(frac_ones_in_cell)), 2))
print("카운트=1 유전자들이 토큰열에서 '코퍼스 중앙값 오름차순'으로 나타나는 세포 비율:", np.mean(frac_sorted))

열 순서를 섞어도 동일: True
카운트 2배(n_counts도 2배)해도 동일: True
카운트만 2배(n_counts 고정)해도 순위 동일: True



하위표본의 비영 항목 중 카운트=1 비율: 0.77, 카운트=2: 0.13, 카운트≥3: 0.10
세포 안에서 카운트=1 유전자의 비율 (중앙값): 0.77
카운트=1 유전자들이 토큰열에서 '코퍼스 중앙값 오름차순'으로 나타나는 세포 비율: 1.0


## 4. 모델 불러오기와 세포 임베딩 추출
- 가중치는 `BertForMaskedLM`으로 저장되어 있다. 임베딩에는 `BertModel`을 쓴다 (MLM 헤드는 무시되고, pooler는 쓰지 않으므로 `add_pooling_layer=False`로 둔다).
- 공식 `EmbExtractor`의 기본값(`emb_layer=-1`)은 **마지막에서 두 번째 층**의 은닉 상태다. `hidden_states[-2]`가 그것이다 (`hidden_states[0]`은 임베딩 층 출력).
- 세포 임베딩 = 유전자 토큰 위치의 평균(패딩과 `<cls>`/`<eos>` 제외). V2는 `<cls>` 위치의 벡터도 따로 저장한다.
- 길이가 비슷한 세포끼리 묶어 패딩 낭비를 줄인다. 패딩이 결과를 바꾸지 않는지 아래에서 확인한다.

In [8]:
@torch.no_grad()
def embed(model, token_lists, special, cls_id=None, eos_id=None, batch=32, layers=(-2, -1)):
    n, H = len(token_lists), model.config.hidden_size
    order = np.argsort([len(t) for t in token_lists])
    out = {("mean", l): np.zeros((n, H), np.float32) for l in layers}
    cls_out = {l: np.zeros((n, H), np.float32) for l in layers} if special else {}
    for b in range(0, n, batch):
        bi = order[b:b + batch]
        ml = max(len(token_lists[i]) for i in bi)
        ids_ = torch.zeros(len(bi), ml, dtype=torch.long)
        for j, i in enumerate(bi):
            ids_[j, :len(token_lists[i])] = torch.from_numpy(token_lists[i])
        ids_ = ids_.to(DEVICE)
        att = ids_ != 0
        hs = model(input_ids=ids_, attention_mask=att.long(), output_hidden_states=True).hidden_states
        gm = att.clone()
        if special:
            gm &= (ids_ != cls_id) & (ids_ != eos_id)
        m = gm.unsqueeze(-1).float()
        for l in layers:
            out[("mean", l)][bi] = ((hs[l] * m).sum(1) / m.sum(1)).float().cpu().numpy()
            if special:
                cls_out[l][bi] = hs[l][:, 0].float().cpu().numpy()
    res = dict(out)
    for l, v in cls_out.items():
        res[("cls", l)] = v
    return res


def load_model(version, random_init=False):
    path = Path(fetch(FILES[version]["model"] + "/config.json")).parent
    fetch(FILES[version]["model"] + "/model.safetensors")
    if random_init:
        torch.manual_seed(SEED)
        return BertModel(BertConfig.from_pretrained(path), add_pooling_layer=False).to(DEVICE).eval()
    return BertModel.from_pretrained(path, add_pooling_layer=False).to(DEVICE).eval()


m1 = load_model("v1")
print("V1 파라미터: %.1fM | 층 %d | hidden %d | attention 구현 %s" % (sum(p.numel() for p in m1.parameters()) / 1e6, m1.config.num_hidden_layers, m1.config.hidden_size, m1.config._attn_implementation))

# 패딩 불변성: 같은 세포를 단독/배치로 넣었을 때 임베딩 차이
test = tokens["v1"][:8]
e_alone = np.vstack([embed(m1, [t], False, batch=1)[("mean", -2)] for t in test])
e_batch = embed(m1, test, False, batch=8)[("mean", -2)]
print("단독 vs 배치(패딩 포함) 최대 절대차: %.2e (임베딩 크기 중앙값 %.2f)" % (np.abs(e_alone - e_batch).max(), np.median(np.abs(e_batch))))

V1 파라미터: 10.2M | 층 6 | hidden 256 | attention 구현 sdpa


단독 vs 배치(패딩 포함) 최대 절대차: 5.96e-08 (임베딩 크기 중앙값 0.02)


### 4.1 세 가지 임베딩 추출 (V1-10M · V1 무작위 초기화 · V2-104M)
**무작위 초기화 V1**은 구조·토큰 사전·입력은 같고 가중치만 사전학습되지 않은 대조군이다. 사전학습의 효과와 "토큰열 자체에 담긴 정보"를 분리하기 위한 것이다.

In [9]:
emb = {}
t0 = time.time()
r = embed(m1, tokens["v1"], False)
for (kind, l), v in r.items():
    emb[f"v1_{kind}_L{l}"] = v
print("V1-10M 완료: %.0f초" % (time.time() - t0))
del m1

m1r = load_model("v1", random_init=True)
t0 = time.time()
r = embed(m1r, tokens["v1"], False)
for (kind, l), v in r.items():
    emb[f"v1rand_{kind}_L{l}"] = v
print("V1 무작위 초기화 완료: %.0f초" % (time.time() - t0))
del m1r

m2 = load_model("v2")
print("V2 파라미터: %.1fM | 층 %d | hidden %d" % (sum(p.numel() for p in m2.parameters()) / 1e6, m2.config.num_hidden_layers, m2.config.hidden_size))
t0 = time.time()
r = embed(m2, tokens["v2"], True, cls_id=tk["v2"].tok["<cls>"], eos_id=tk["v2"].tok["<eos>"])
for (kind, l), v in r.items():
    emb[f"v2_{kind}_L{l}"] = v
print("V2-104M 완료: %.0f초" % (time.time() - t0))
del m2
if DEVICE.type == "mps":
    torch.mps.empty_cache()

print({k: v.shape for k, v in emb.items()})
print("NaN 포함:", {k: bool(np.isnan(v).any()) for k, v in emb.items() if np.isnan(v).any()} or "없음")

V1-10M 완료: 71초


V1 무작위 초기화 완료: 71초


V2 파라미터: 103.8M | 층 12 | hidden 768


V2-104M 완료: 705초


{'v1_mean_L-2': (10400, 256), 'v1_mean_L-1': (10400, 256), 'v1rand_mean_L-2': (10400, 256), 'v1rand_mean_L-1': (10400, 256), 'v2_mean_L-2': (10400, 768), 'v2_mean_L-1': (10400, 768), 'v2_cls_L-2': (10400, 768), 'v2_cls_L-1': (10400, 768)}
NaN 포함: 없음


### 4.2 저장 — 노트북 02·03이 읽는다
하위표본 카운트(원시)와 메타데이터를 `pbmc_sub.h5ad`로, 임베딩과 토큰열을 `.npz`/`.pkl`로 저장한다 (모두 `.cache/` 아래, git 제외).

In [10]:
sub = ad.AnnData(X=counts.astype(np.float32), obs=obs.copy(), var=pd.DataFrame(index=var_ids))
sub.obs_names = [f"cell{i}" for i in range(sub.n_obs)]
sub.var["gene_symbol"] = adata.raw.var["feature_name"].values
sub.write_h5ad(SC / "pbmc_sub.h5ad")
np.savez_compressed(SC / "geneformer_embeddings.npz", **emb)
with open(SC / "geneformer_tokens.pkl", "wb") as f:              # 우리가 만든 데이터라 pickle이어도 되지만, 읽을 때도 노트북 02·03에서 허용 목록으로 읽는다
    pickle.dump({v: [t.astype(np.int32) for t in tokens[v]] for v in tokens}, f)
print("저장:", sorted(p.name for p in SC.iterdir()))

저장: ['cxg_cross_sectional.h5ad', 'geneformer_embeddings.npz', 'geneformer_tokens.pkl', 'pbmc_sub.h5ad', 'wilk2020_pbmc.h5ad']


## 5. 사전학습이 무언가 배웠는가 — 마스크 유전자 예측 점검
하위표본에서 300개 세포를 뽑아 유전자 토큰의 15%를 `<mask>`로 가리고, 모델이 원래 유전자를 얼마나 맞히는지 본다. 비교 대상은 다음과 같다.
- (a) **전역 빈도 prior**: 어느 위치든 "많이 검출되는 유전자"를 제시한다. (b) **무작위 초기화** 모델. (c) 우연 수준. (e) **카운트≥2 빈도 prior**: 정답이 카운트≥2인 위치에서, 나머지 세포에서 카운트≥2로 나타난 빈도 순 상위 후보를 제시한다(위치·문맥 무관).
- (d) **순서 구간 baseline**: 3.1절에서 본 대로 카운트=1 유전자들은 **코퍼스 중앙값 오름차순**으로 늘어선다. 가려진 위치의 좌우에서 가장 가까운 안 가려진 이웃이 둘 다 카운트=1이면, 정답의 중앙값은 두 이웃의 중앙값 사이에 있다. 그 구간에 속하는 유전자 중 **검출 빈도가 높은 순**으로 제시한다. 유전자 간 공발현 모델은 쓰지 않지만, **해당 세포의 이웃 토큰(위치와 중앙값)과 정답·이웃의 카운트 정보는 쓴다**.

위치를 "정답이 카운트=1이고 좌우 이웃도 카운트=1인 위치(순서로 후보가 좁혀지는 위치)"와 "정답이 카운트≥2인 위치"로 나눠 본다.
**주의**: (d)는 "정답이 카운트=1이고 이웃도 카운트=1"이라는 **정답 쪽 정보를 미리 안다**(모델은 모른다). 그래서 (d)와 (e)는 위치 분류에 정답 쪽 정보를 쓰는 *오라클형* baseline이다. 모델이 (d)보다 낮다고 해서 모델이 못한다는 뜻이 아니라, 반대로 (d)가 높다는 것이 "이 과제의 점수에서 순서 구조가 차지하는 몫"을 보여 준다.
사전학습 코퍼스에 이 데이터셋(2020)이 포함되었는지는 **확인하지 못했다** (코퍼스 구성 미확인). 포함되었다면 점수가 낙관적일 수 있다.

In [11]:
@torch.no_grad()
def mlm_predict(model, token_lists, special_ids, mask_id, rng, p=0.15, batch=16, topk=10):
    """각 세포에서 유전자 토큰의 p를 가리고 (세포 인덱스, 위치, 정답, 상위 topk 예측)을 돌려준다."""
    rec = []
    for b in range(0, len(token_lists), batch):
        chunk = token_lists[b:b + batch]
        ml = max(len(t) for t in chunk)
        ids_ = torch.zeros(len(chunk), ml, dtype=torch.long)
        for j, t in enumerate(chunk):
            ids_[j, :len(t)] = torch.from_numpy(t.astype(np.int64))
        gene_pos = (ids_ != 0) & ~torch.isin(ids_, torch.tensor(list(special_ids)))
        masked = gene_pos & torch.from_numpy(rng.random(ids_.shape) < p)
        inp = ids_.clone()
        inp[masked] = mask_id
        h = model.bert(input_ids=inp.to(DEVICE), attention_mask=(ids_ != 0).long().to(DEVICE)).last_hidden_state
        top = model.cls(h[masked.to(DEVICE)]).topk(topk, dim=-1).indices.cpu().numpy()
        ci, pos = torch.nonzero(masked, as_tuple=True)
        rec.append((ci.numpy() + b, pos.numpy(), ids_[masked].numpy(), top))
    return [np.concatenate(x) for x in zip(*rec)], masked_all(rec)


def masked_all(rec):
    return sum(len(r[0]) for r in rec)


def load_mlm(version, random_init=False):
    path = Path(fetch(FILES[version]["model"] + "/config.json")).parent
    fetch(FILES[version]["model"] + "/model.safetensors")
    if random_init:
        torch.manual_seed(SEED)
        return BertForMaskedLM(BertConfig.from_pretrained(path)).to(DEVICE).eval()
    return BertForMaskedLM.from_pretrained(path).to(DEVICE).eval()


def topk_acc(top, true, ks=(1, 10)):
    return {k: float((top[:, :k] == true[:, None]).any(1).mean()) if len(true) else float("nan") for k in ks}


eval_idx = np.random.default_rng(SEED).choice(len(obs), 300, replace=False)
rest_idx = np.setdiff1d(np.arange(len(obs)), eval_idx)
rows, mlm_store = [], {}
for v in ["v1", "v2"]:
    t = tk[v]
    vocab = len(t.tok)
    specials = {0, 1} | ({t.tok["<cls>"], t.tok["<eos>"]} if t.special else set())
    # 평가 세포의 토큰/카운트 (3절 구현과 토큰이 일치하는지도 확인)
    Xc_e, _ = t.collapse(counts[eval_idx], var_ids)
    tok_e, cnt_e = t.to_tokens(t.normalize_collapsed(Xc_e, n_counts[eval_idx]), Xc_e)
    assert all(np.array_equal(a, tokens[v][i]) for a, i in zip(tok_e, eval_idx))
    # 사전: 토큰 -> 코퍼스 중앙값, 나머지 세포에서의 검출 빈도(df)
    med_tok = np.full(vocab, np.nan)
    med_tok[t.token_ids] = t.median
    df = np.bincount(np.concatenate([tokens[v][i] for i in rest_idx]), minlength=vocab).astype(float)
    df[list(specials)] = 0
    # 카운트≥2 baseline: 나머지 세포에서 카운트≥2로 나타난 빈도 순 (위치·문맥 무관)
    Xc_r, _ = t.collapse(counts[rest_idx], var_ids)
    df2 = np.zeros(vocab)
    df2[t.token_ids] = np.asarray((Xc_r >= 2).sum(0)).ravel()
    top2 = np.argsort(-df2)[:10]
    gene_tokens = np.array(sorted(t.token_ids, key=lambda x: med_tok[x]))
    gene_med_sorted = med_tok[gene_tokens]
    for name, rand in [("사전학습", False), ("무작위 초기화", True)]:
        if v == "v2" and rand:
            continue
        mdl = load_mlm(v, rand)
        (ci, pos, true, top), nmask = mlm_predict(mdl, tok_e, specials, t.tok["<mask>"], np.random.default_rng(SEED))
        del mdl
        if DEVICE.type == "mps":
            torch.mps.empty_cache()
        # 위치 분류: 정답 카운트, 좌우 이웃(안 가려진 가장 가까운 유전자 토큰)의 카운트
        masked_set = {}
        for c, p_ in zip(ci, pos):
            masked_set.setdefault(c, set()).add(p_)
        grp = np.zeros(len(ci), dtype=object)
        base_top = np.full((len(ci), 10), -1)
        cand_sizes = np.zeros(len(ci))
        for n_, (c, p_, tr) in enumerate(zip(ci, pos, true)):
            cnt = cnt_e[c]
            if cnt[p_] != 1:
                grp[n_] = "정답 카운트≥2"
                continue
            ms = masked_set[c]
            lo, hi = p_ - 1, p_ + 1
            while lo >= 0 and (lo in ms or tok_e[c][lo] in specials):
                lo -= 1
            while hi < len(cnt) and (hi in ms or tok_e[c][hi] in specials):
                hi += 1
            if lo < 0 or hi >= len(cnt) or cnt[lo] != 1 or cnt[hi] != 1:
                grp[n_] = "기타(카운트=1이나 이웃이 1 아님)"
                continue
            grp[n_] = "순서 구간 위치"
            m_lo, m_hi = sorted([med_tok[tok_e[c][lo]], med_tok[tok_e[c][hi]]])
            cand = gene_tokens[np.searchsorted(gene_med_sorted, m_lo, side="left"):np.searchsorted(gene_med_sorted, m_hi, side="right")]
            cand_sizes[n_] = len(cand)
            base_top[n_, :min(10, len(cand))] = cand[np.argsort(-df[cand], kind="stable")[:10]]
        gtop = np.argsort(-df)[:10]
        rows.append(dict(모델=f"{v.upper()} {name}", 구분="전체", 위치수=len(true), **{f"top{k}": a for k, a in topk_acc(top, true).items()},
                         전역빈도prior_top10=float(np.isin(true, gtop).mean()), 우연_top1=1 / vocab))
        for g in ["순서 구간 위치", "정답 카운트≥2", "기타(카운트=1이나 이웃이 1 아님)"]:
            m = grp == g
            r_ = dict(모델=f"{v.upper()} {name}", 구분=g, 위치수=int(m.sum()), **{f"top{k}": a for k, a in topk_acc(top[m], true[m]).items()})
            if g == "정답 카운트≥2":
                r_.update(**{"카운트≥2빈도prior_top1": float(np.isin(true[m], top2[:1]).mean()), "카운트≥2빈도prior_top10": float(np.isin(true[m], top2).mean())})
            if g == "순서 구간 위치" and not rand:
                bb = topk_acc(base_top[m], true[m])
                r_.update(순서구간baseline_top1=bb[1], 순서구간baseline_top10=bb[10], 후보수_중앙값=float(np.median(cand_sizes[m])))
            rows.append(r_)
res_mlm = pd.DataFrame(rows).set_index(["모델", "구분"])
print(res_mlm.round(4).to_string())

                                   위치수    top1   top10  전역빈도prior_top10  우연_top1  순서구간baseline_top1  순서구간baseline_top10  후보수_중앙값  카운트≥2빈도prior_top1  카운트≥2빈도prior_top10
모델         구분                                                                                                                                                          
V1 사전학습    전체                    40298  0.2129  0.7174           0.0100      0.0                NaN                 NaN      NaN                NaN                 NaN
           순서 구간 위치              25086  0.2177  0.7617              NaN      NaN             0.1209              0.7043     47.0                NaN                 NaN
           정답 카운트≥2               8817  0.2057  0.6039              NaN      NaN                NaN                 NaN      NaN             0.0045              0.0423
           기타(카운트=1이나 이웃이 1 아님)   6395  0.2038  0.6998              NaN      NaN                NaN                 NaN      NaN                NaN             

## 6. 해석과 한계
아래는 이 실행(seed 0, Mac MPS fp32)에서 읽은 내용이다. **출력을 직접 확인한다.**

**데이터와 토큰화**
- 전체 데이터(44,721세포)는 세포당 총 카운트 중앙값 1,508, 검출 유전자 중앙값 919로 얕다(1.1절). 하위표본에서도 비영 항목의 **77%가 카운트 1**이고, 그 유전자들은 모든 세포에서 *코퍼스 중앙값 오름차순*으로 늘어선다(확인한 세포 300개 모두). 따라서 이 데이터에서 세포마다 다른 정보는 사실상 (1) **어떤 유전자가 검출되었는가**와 (2) 카운트≥2 유전자가 순위열의 어디에 끼어드는가다. 노트북 03에서 이 의존성을 ablation으로 시험한다.
- 토큰열 길이는 평균 약 910, 최대 1,873으로 V1의 2,048·V2의 4,094 한도에 걸리지 않았다. 이 데이터에서 "문맥 길이 절단"은 쟁점이 아니다.
- 어휘와 연결되는 열은 24,505개 중 약 70%이지만 **총 카운트의 약 90%**가 거기에 있다. 나머지 열(주로 비코딩 등)은 토큰이 되지 못해 버려진다. 그 열에 있던 정보는 모델에 전혀 전달되지 않는다.
- 제공된 `nCount_RNA`는 `raw.X` 합과 맞지 않는다(상관 0.53). 그러나 `n_counts`는 한 세포 안의 모든 유전자를 **같은 양수로** 나누므로 순위·토큰열을 바꾸지 못한다(3.1절의 스케일 불변 점검과 같은 원리; `nCount_RNA`로 바꿔 직접 확인하는 것은 과제 1c). 즉 이 불일치는 *토큰화 결과에는 무해*하며, 깊이 정규화된 값 자체를 쓰는 분석(PCA 등)에서만 영향을 줄 수 있다.
- 질병 상태가 기증자와 완전히 겹친다. 그래서 이후 "배치=기증자"로 평가할 때 배치와 생물학이 분리되지 않는다.

**구현 점검**
- 열 순서·카운트 스케일에 불변이고 패딩 유무에 따른 임베딩 차이는 6e-8 수준으로 구현이 의도대로 동작한다. 다만 **공식 패키지 출력과 토큰 단위로 대조하지는 못했다.**
- 속도: Mac MPS fp32에서 10,400세포 기준 V1-10M 68초, V2-104M 673초(이전 실행 816초; 약 10~12배). 다른 작업과 동시에 실행하면 달라진다. 두 번 실행한 임베딩은 비트 단위로 같았다(해시 비교).

**마스크 유전자 예측 점검 (300세포, 약 4만 마스크 위치)**
- 사전학습 모델의 top-10 정확도는 V1 71.7%, V2 75.2%, **무작위 초기화는 0.03%**다. 사전학습이 *무언가를* 학습했다는 최소 확인이다.
- 그러나 "순서 구간 위치"(전체의 약 60%)에서, 가려진 유전자의 중앙값이 좌우 이웃 중앙값 사이에 있다는 사실과 검출 빈도만 쓰는 baseline이 이미 top-10 70.4%(V1)/71.9%(V2)에 도달한다(모델 76.2%/78.8%). 이 baseline은 유전자 간 공발현 모델은 쓰지 않지만 해당 세포의 이웃 토큰과 정답·이웃의 카운트 정보를 쓰는 오라클형이라 모델과 같은 조건의 비교는 아니다. 그래도 **MLM top-k 정확도만으로 생물학적 지식을 주장할 수 없다**는 근거는 된다. top-1은 모델 21.8%/23.7% 대 baseline 12.1%/12.2%로 모델이 더 높다.
- 반면 정답이 카운트≥2인 위치에서는 모델 top-10이 V1 60.4%, V2 68.7%인데, 위치와 무관하게 "카운트≥2로 자주 나오는 유전자"를 제시하는 빈도 prior는 4.2%/4.0%에 그친다. 이 위치의 점수는 순서 구조(중앙값 정렬)로는 설명되지 않으므로 모델이 세포 문맥을 일부 쓴다는 **신호**다. 다만 순위 위치(앞쪽일수록 고발현)를 쓰는 baseline은 시험하지 않았으므로 문맥 사용을 확정하지는 않는다.
- 이 데이터셋(2020)이 사전학습 코퍼스에 포함되었는지는 확인하지 못했다. 점검은 300세포·단일 데이터셋·단일 마스킹 시드다.

**다음**: 노트북 02에서 이 임베딩이 세포 유형을 보존하고 기증자를 섞는지 PCA·Harmony·scVI와 비교한다.